# Faisal H2-PRISM pipeline walkthrough

This notebook accompanies the reproducible Python pipeline. It makes the F1-F13 workstream and data flow inspectable one stage at a time, walks through Fold 1 in detail, then fits and evaluates all three outer folds using the same functions as the command-line runner. The Python modules remain the source of truth; this notebook does not reimplement the estimators.

**Dataset note:** the Parquet files under `data/raw/` are the local H2-SimNet benchmark. They are real files used in the run, but the benchmark data are simulated hydrogen-blend data, not field measurements.

**Evaluation caveats:** the supplied folds have base-family overlap across train/test; one scenario with topology recorded as `Unknown` remains in each training split; and abnormal windows are selected using audited event intervals. Results are cross-topology event-attribution results, not unseen-family or prospective event-detection results.

## Before running

Use the repository Python environment and install [`requirements.txt`](../requirements.txt) if needed. The full benchmark must be present under `data/raw/`; if it is missing, use [`setup_data.py`](../setup_data.py) (which invokes the downloader) or [`scripts/download_data.py`](../scripts/download_data.py). The feature and model cells below use the benchmark Parquet files, not the dashboard demo data.

## F1-F13 task map

| Task | What to inspect in this notebook | Main implementation/evidence |
|---|---|---|
| F1 Repository audit | Ownership boundaries and prior-art classification | `02_methodology/FAISAL_REPOSITORY_IMPLEMENTATION_AUDIT.md` |
| F2 Problem formulation | Four-class target and evaluation limits | `02_methodology/DIAGNOSTIC_PROBLEM_FORMULATION.md` |
| F3 Preprocessing/windows/features | Raw rows, eligibility, windows, and sensor summaries | `src/h2prism/windows.py`, `WINDOW_MANIFEST.csv`, `FEATURE_TABLE.csv` |
| F4 Grouped evaluation | Outer folds, topology groups, and family overlap | `src/h2prism/evaluation.py`, `FOLD_VALIDATION_REPORT.csv` |
| F5 Baseline | Family-weighted random forest and predictions | `BASELINE_RESULTS.csv`, `OUTER_PREDICTIONS.csv` |
| F6 Calibration | Inner-OOF calibration selection and outer metrics | `CALIBRATION_RESULTS.csv` |
| F7 Empirical support D(x) | Family-excluded neighbors and q/D values | `D_STATISTIC_DESIGN_DECISION.md`, `OUTER_PREDICTIONS.csv` |
| F8 Comparators | B, B2, B3, C-D, C, and D-only scores | `ABLATION_RESULTS.csv`, `RISK_COVERAGE_CURVES.csv` |
| F9 Selective policy | Specific, ambiguity-set, and unresolved outcomes | `POLICY_OUTCOMES.csv`, `OUTER_PREDICTIONS.csv` |
| F10 Primary comparison | Matched-coverage and deployed-threshold results | `MAIN_EXPERIMENT_REPORT.md`, `SELECTIVE_EVALUATION_RESULTS.csv` |
| F11 Statistical validation | Family-level uncertainty and paired comparisons | `STATISTICAL_VALIDATION.csv`, `PER_FAMILY_RESULTS.csv` |
| F12 Integration freeze | Accepted settings, interfaces, and generated artifacts | `configs/frozen_evaluation_config.yaml`, `TECHNICAL_ARCHITECTURE_FREEZE.md` |
| F13 Testing/reproducibility | Focused tests, hashes, and runtime versions | `04_validation/TEST_RESULTS.md`, `RESULTS_PROVENANCE.csv` |

The plan's F1-F13 deliverables did not require a Jupyter notebook specifically. This is an added walkthrough; the `.py` modules and versioned results remain canonical.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "h2prism").is_dir():
    parent = PROJECT_ROOT.parent
    if parent == PROJECT_ROOT:
        raise FileNotFoundError("Open this notebook from the H2-PRISM repository.")
    PROJECT_ROOT = parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
DATA_ROOT = PROJECT_ROOT / "data" / "raw"
RESULTS_ROOT = PROJECT_ROOT / "03_results"
print(f"Project: {PROJECT_ROOT}")
print(f"Raw data directory exists: {DATA_ROOT.is_dir()}")
print(f"Current generated results exist: {RESULTS_ROOT.is_dir()}")

Project: C:\Users\faisa\OneDrive\سطح المكتب\CURRENT_PROJECTS\H2-PRISM
Raw data directory exists: True
Current generated results exist: True


In [2]:
import json
import importlib
import numpy as np
import pandas as pd
import sys
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, f1_score

for module_name in tuple(sys.modules):
    if module_name == "h2prism" or module_name.startswith("h2prism."):
        del sys.modules[module_name]
importlib.invalidate_caches()

from h2prism.windows import SENSOR_CHANNELS, TARGET_CLASSES, build_window_dataset
from h2prism.evaluation import (
    FOLDS_PATH,
    MANIFEST_PATH,
    add_distinguishability_score,
    aligned_probabilities,
    apply_diagnostic_policy,
    compute_empirical_support,
    fit_baseline,
    inner_oof_predictions,
    load_outer_fold,
    model_feature_columns,
    probability_indicators,
    probability_metrics,
    threshold_for_coverage,
)
from h2prism.experiments import (
    DEFAULT_CONFIG,
    DEFAULT_OUTPUT_DIR,
    SCORE_COLUMNS,
    _oof_calibration_choice,
    _read_config,
    _selective_metrics,
    run_experiments,
)
import h2prism.experiments as experiment_module

config = _read_config(DEFAULT_CONFIG)
FOLD_ID = int(config["outer_folds"][0])
TARGET_COVERAGE = float(config["target_specific_coverage"])
print(f"Pipeline source: {Path(experiment_module.__file__).resolve()}")
print(f"Frozen config: {Path(DEFAULT_CONFIG).resolve()}")
print(f"Frozen config {config['config_version']}; walkthrough outer fold {FOLD_ID}")
print(f"Target classes: {TARGET_CLASSES}")

Pipeline source: C:\Users\faisa\OneDrive\سطح المكتب\CURRENT_PROJECTS\H2-PRISM\src\h2prism\experiments.py
Frozen config: C:\Users\faisa\OneDrive\سطح المكتب\CURRENT_PROJECTS\H2-PRISM\configs\frozen_evaluation_config.yaml
Frozen config 1.0.2; walkthrough outer fold 1
Target classes: ('NORMAL', 'LEAK', 'COMPRESSOR_FAULT', 'COMPRESSOR_DELAY')


## F1, F2, and F12 — ownership, question, and frozen settings

F1 records which artifacts are reused read-only versus implemented for Faisal. F2 defines four-class attribution on eligible windows. F12 freezes the run settings before accepting outer results. The actual task map is above; this cell exposes the settings the code will use.

In [3]:
config_summary = {
    "config_version": config["config_version"],
    "outer_folds": config["outer_folds"],
    "window_size_samples": config["window_size_samples"],
    "stride_samples": config["stride_samples"],
    "sampling_rate_hz": config["sampling_rate_hz"],
    "d_neighbors": config["d_neighbors"],
    "target_specific_coverage": config["target_specific_coverage"],
    "bootstrap_replicates": config["bootstrap_replicates"],
    "model": config["model"],
    "calibration": config["calibration"],
    "policy": config["policy"],
}
display(pd.Series(config_summary, name="frozen value"))

config_version                                                          1.0.2
outer_folds                                                         [1, 2, 3]
window_size_samples                                                        60
stride_samples                                                             60
sampling_rate_hz                                                            1
d_neighbors                                                                25
target_specific_coverage                                                  0.8
bootstrap_replicates                                                     1000
model                       {'estimator': 'RandomForestClassifier', 'n_est...
calibration                 {'candidates': ['none', 'sigmoid', 'isotonic']...
policy                      {'ambiguity_delta': 0.15, 'max_candidates': 3,...
Name: frozen value, dtype: object

## F4 — inspect the supplied outer folds

The folds and provisional manifest are not changed. The loader deduplicates mirror records to one canonical Parquet per scenario, verifies label/topology metadata, and calculates family overlap. The `Unknown` topology scenario remains in training; the inner calibration split is a two-fold GroupKFold over the three training topology values, not leave-one-group-out for each one. The data also have only one family for three classes, so fully family-disjoint four-class evaluation is impossible without more data.

In [4]:
fold_assignments = pd.read_csv(FOLDS_PATH)
print("Supplied fold table:")
display(fold_assignments.head())

fold_overview = []
all_scenarios = fold_assignments.drop_duplicates("Scenario_ID")
family_counts = all_scenarios.groupby("target_class")["Scenario_Family_ID"].nunique()
family_counts_text = ";".join(f"{name}={family_counts.get(name, 0)}" for name in TARGET_CLASSES)
for fold_id, fold_rows in fold_assignments.groupby("outer_fold_id"):
    train_rows = fold_rows[fold_rows["split_role"] == "train"].drop_duplicates("Scenario_ID")
    test_rows = fold_rows[fold_rows["split_role"] == "test"].drop_duplicates("Scenario_ID")
    train_families = set(train_rows["Scenario_Family_ID"].astype(str))
    test_families = set(test_rows["Scenario_Family_ID"].astype(str))
    overlap = sorted(train_families & test_families)
    fold_overview.append({
        "outer_fold": int(fold_id),
        "train_scenarios": train_rows["Scenario_ID"].nunique(),
        "test_scenarios": test_rows["Scenario_ID"].nunique(),
        "train_topologies": ";".join(sorted(train_rows["topology"].astype(str).unique())),
        "test_topologies": ";".join(sorted(test_rows["topology"].astype(str).unique())),
        "family_overlap_count": len(overlap),
        "family_overlap": ";".join(overlap),
        "family_counts_by_class": family_counts_text,
        "single_family_classes": ";".join(family_counts[family_counts < 2].index),
    })
display(pd.DataFrame(fold_overview))

train_records, test_records, fold1_diagnostics = load_outer_fold(FOLD_ID)
print(f"Loaded {len(train_records)} train and {len(test_records)} test scenarios")
display(pd.Series(fold1_diagnostics))

Supplied fold table:


,outer_fold_id,fold_name,split_role,Scenario_ID,Scenario_Family_ID,Topology_Family_ID,file,topology,target_class,train_classes_available,test_classes_available,zero_leakage_assertion_passed
0,1,Fold_1_Test_DOWNSTREAM,train,SCEN_CENTER_TRANSIENT_LEAKG4,FAM_LEAK_G4,FAM_LEAK_G4_CENTER,H2-SimNet-Dataset/H2-SimNet-clean/anomalous-sc...,Center,LEAK,COMPRESSOR_DELAY;COMPRESSOR_FAULT;LEAK;NORMAL,COMPRESSOR_DELAY;COMPRESSOR_FAULT;LEAK;NORMAL,True
1,1,Fold_1_Test_DOWNSTREAM,train,SCEN_CENTER_STEADY_STATE_BROKENC1_STRONG,FAM_COMPRESSOR_FAULT_C1,FAM_COMPRESSOR_FAULT_C1_CENTER,H2-SimNet-Dataset/H2-SimNet-clean/anomalous-sc...,Center,COMPRESSOR_FAULT,COMPRESSOR_DELAY;COMPRESSOR_FAULT;LEAK;NORMAL,COMPRESSOR_DELAY;COMPRESSOR_FAULT;LEAK;NORMAL,True
2,1,Fold_1_Test_DOWNSTREAM,train,SCEN_CENTER_STEADY_STATE_BROKENC1_WEAK,FAM_COMPRESSOR_FAULT_C1,FAM_COMPRESSOR_FAULT_C1_CENTER,H2-SimNet-Dataset/H2-SimNet-clean/anomalous-sc...,Center,COMPRESSOR_FAULT,COMPRESSOR_DELAY;COMPRESSOR_FAULT;LEAK;NORMAL,COMPRESSOR_DELAY;COMPRESSOR_FAULT;LEAK;NORMAL,True
3,1,Fold_1_Test_DOWNSTREAM,train,SCEN_CENTER_STEADY_STATE_LEAKG3,FAM_LEAK_G3,FAM_LEAK_G3_CENTER,H2-SimNet-Dataset/H2-SimNet-clean/anomalous-sc...,Center,LEAK,COMPRESSOR_DELAY;COMPRESSOR_FAULT;LEAK;NORMAL,COMPRESSOR_DELAY;COMPRESSOR_FAULT;LEAK;NORMAL,True
4,1,Fold_1_Test_DOWNSTREAM,train,SCEN_CENTER_STEADY_STATE_LEAKG3G4_STRONG,FAM_LEAK_G3G4_COMBINED,FAM_LEAK_G3G4_COMBINED_CENTER,H2-SimNet-Dataset/H2-SimNet-clean/anomalous-sc...,Center,LEAK,COMPRESSOR_DELAY;COMPRESSOR_FAULT;LEAK;NORMAL,COMPRESSOR_DELAY;COMPRESSOR_FAULT;LEAK;NORMAL,True


,outer_fold,train_scenarios,test_scenarios,train_topologies,test_topologies,family_overlap_count,family_overlap,family_counts_by_class,single_family_classes
0,1,31,15,Center;Unknown;Upstream,Downstream,9,FAM_COMPRESSOR_DELAY_C1;FAM_COMPRESSOR_FAULT_C...,NORMAL=1;LEAK=6;COMPRESSOR_FAULT=1;COMPRESSOR_...,COMPRESSOR_DELAY;COMPRESSOR_FAULT;NORMAL
1,2,31,15,Downstream;Unknown;Upstream,Center,9,FAM_COMPRESSOR_DELAY_C1;FAM_COMPRESSOR_FAULT_C...,NORMAL=1;LEAK=6;COMPRESSOR_FAULT=1;COMPRESSOR_...,COMPRESSOR_DELAY;COMPRESSOR_FAULT;NORMAL
2,3,31,15,Center;Downstream;Unknown,Upstream,9,FAM_COMPRESSOR_DELAY_C1;FAM_COMPRESSOR_FAULT_C...,NORMAL=1;LEAK=6;COMPRESSOR_FAULT=1;COMPRESSOR_...,COMPRESSOR_DELAY;COMPRESSOR_FAULT;NORMAL


Loaded 31 train and 15 test scenarios


outer_fold                                                                1
train_scenarios                                                          31
test_scenarios                                                           15
train_topologies                                [Center, Unknown, Upstream]
test_topologies                                                [Downstream]
train_families                                                            9
test_families                                                             9
family_overlap_count                                                      9
family_overlap            FAM_COMPRESSOR_DELAY_C1;FAM_COMPRESSOR_FAULT_C...
family_independent                                                    False
family_counts_by_class    NORMAL=1;LEAK=6;COMPRESSOR_FAULT=1;COMPRESSOR_...
single_family_classes              NORMAL;COMPRESSOR_FAULT;COMPRESSOR_DELAY
train_classes                 COMPRESSOR_DELAY;COMPRESSOR_FAULT;LEAK;NORMAL
test_classes

## F3 — start from an actual Parquet scenario

Each record returned by `load_outer_fold` pairs one manifest row with the corresponding Parquet DataFrame. The manifest supplies scenario/family identity, mapped target, eligibility, topology, and event boundaries; the Parquet supplies timestamped sensor readings.

In [5]:
sample_metadata, sample_raw = train_records[0]
raw_path = DATA_ROOT / str(sample_metadata["file"])
print(f"Reading: {raw_path}")
print(f"Rows/columns: {sample_raw.shape}")
display(sample_metadata[[
    "Scenario_ID", "Scenario_Family_ID", "target_class", "event_label_raw",
    "event_onset_s", "event_offset_s", "topology", "eligible_primary", "file"
]])
display(sample_raw.head())
missing_sensor_channels = sorted(set(SENSOR_CHANNELS) - set(sample_raw.columns))
print(f"Expected sensor channels: {len(SENSOR_CHANNELS)}; missing: {missing_sensor_channels}")

Reading: C:\Users\faisa\OneDrive\سطح المكتب\CURRENT_PROJECTS\H2-PRISM\data\raw\H2-SimNet-Dataset\H2-SimNet-clean\anomalous-scenarios\parquet\anom_gestcent\start_transient\leakG3.parquet
Rows/columns: (20000, 24)


Scenario_ID                          SCEN_CENTER_INITIAL_RAMP_UP_LEAKG3
Scenario_Family_ID                                          FAM_LEAK_G3
target_class                                                       LEAK
event_label_raw                                                 Leak_G3
event_onset_s                                                     100.0
event_offset_s                                                  19999.0
topology                                                         Center
eligible_primary                                                   True
file                  H2-SimNet-Dataset/H2-SimNet-clean/anomalous-sc...
Name: 74, dtype: object

,PS1,PS2,PS3,PS4,PS5,PS6,PS7,PS8,PS9,PS10,...,MFS3,MFS4,label_anomaly,Leak,CompressorFault,CompressorDelay,leak_G3,leak_G4,leak_G5,leak_G6
Timestamp,,,,,,,,,,,,,,,,,,,,,
0.0,3.733928,3.704757,3.393856,3.340737,0.1,0.74,3.404757,3.74,3.340737,2.340737,...,4.880885e-07,21.983785,0,0,0,0,0,0,0,0
1.0,3.733928,3.704757,3.393856,3.340737,0.1,0.74,3.404757,3.74,3.340737,2.340737,...,4.880885e-07,21.983785,0,0,0,0,0,0,0,0
2.0,3.733928,3.704757,3.393856,3.340737,0.1,0.74,3.404757,3.74,3.340737,2.340737,...,4.880885e-07,21.983785,0,0,0,0,0,0,0,0
3.0,3.733928,3.704757,3.393856,3.340737,0.1,0.74,3.404757,3.74,3.340737,2.340737,...,4.880885e-07,21.983785,0,0,0,0,0,0,0,0
4.0,3.730439,3.704741,3.393856,3.340737,0.1,0.73,3.404741,3.73,3.340737,2.340737,...,4.879269e-07,21.983785,0,0,0,0,0,0,0,0


Expected sensor channels: 16; missing: []


## F3 — turn samples into eligible windows

The frozen setting is 60 samples with a 60-sample stride at 1 Hz. Abnormal windows are eligible only when wholly within the audited event interval. Boundary-crossing, pre/post-event, compound, and other excluded windows stay visible in the manifest but do not enter the feature table. A normal scenario contributes baseline windows.

In [6]:
example_records = []
for wanted_class in ("NORMAL", "LEAK"):
    match = next((record for record in train_records
                  if str(record[0]["target_class"]) == wanted_class), None)
    if match is not None:
        example_records.append(match)

example_manifest, example_features = build_window_dataset(
    example_records,
    window_size=int(config["window_size_samples"]),
    stride=int(config["stride_samples"]),
)
display(example_manifest[[
    "window_id", "Scenario_ID", "label_mapped", "start_time", "end_time",
    "boundary_status", "eligible_primary", "exclusion_reason", "tail_samples_omitted"
]].head(15))
display(example_manifest.groupby(
    ["label_mapped", "boundary_status", "eligible_primary"], dropna=False
).size().rename("windows").to_frame())
print(f"Eligible feature rows from these example scenarios: {len(example_features)}")

,window_id,Scenario_ID,label_mapped,start_time,end_time,boundary_status,eligible_primary,exclusion_reason,tail_samples_omitted
0,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CEN...,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CENTER,NORMAL,0.0,59.0,DURING_EVENT,True,,20
1,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CEN...,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CENTER,NORMAL,60.0,119.0,DURING_EVENT,True,,20
2,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CEN...,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CENTER,NORMAL,120.0,179.0,DURING_EVENT,True,,20
3,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CEN...,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CENTER,NORMAL,180.0,239.0,DURING_EVENT,True,,20
4,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CEN...,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CENTER,NORMAL,240.0,299.0,DURING_EVENT,True,,20
5,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CEN...,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CENTER,NORMAL,300.0,359.0,DURING_EVENT,True,,20
6,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CEN...,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CENTER,NORMAL,360.0,419.0,DURING_EVENT,True,,20
7,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CEN...,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CENTER,NORMAL,420.0,479.0,DURING_EVENT,True,,20
8,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CEN...,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CENTER,NORMAL,480.0,539.0,DURING_EVENT,True,,20
9,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CEN...,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CENTER,NORMAL,540.0,599.0,DURING_EVENT,True,,20


windows
label_mapped boundary_status   eligible_primary         
LEAK         BOUNDARY_CROSSING False                   1
             DURING_EVENT      True                  331
             PRE_EVENT         False                   1
NORMAL       DURING_EVENT      True                  333

Eligible feature rows from these example scenarios: 664


## F3 — inspect the model inputs

For each of 16 pressure/mass-flow sensors, the implementation computes mean, standard deviation, 5th-to-95th percentile range, and first-difference mean/std (80 features total). Identifiers, labels, phase, topology, and time are retained only as metadata/targets, not estimator features.

In [7]:
example_feature_columns = model_feature_columns(example_features)
print(f"Sensor channels: {len(SENSOR_CHANNELS)}; feature columns: {len(example_feature_columns)}")
print("First feature names:", example_feature_columns[:10])
display(example_features[[
    "window_id", "Scenario_ID", "Scenario_Family_ID", "label_mapped",
    *example_feature_columns[:5]
]].head())
print("Non-sensor metadata in model feature list:", [
    name for name in example_feature_columns
    if name in {"Scenario_ID", "Scenario_Family_ID", "label_mapped", "topology", "Timestamp"}
])

Sensor channels: 16; feature columns: 80
First feature names: ['PS1__mean', 'PS1__std', 'PS1__q05_q95_range', 'PS1__diff_mean', 'PS1__diff_std', 'PS2__mean', 'PS2__std', 'PS2__q05_q95_range', 'PS2__diff_mean', 'PS2__diff_std']


,window_id,Scenario_ID,Scenario_Family_ID,label_mapped,PS1__mean,PS1__std,PS1__q05_q95_range,PS1__diff_mean,PS1__diff_std
0,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CEN...,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CENTER,FAM_NORMALITY,NORMAL,4.686641,1.007091,2.485629,0.042912,0.050290
1,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CEN...,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CENTER,FAM_NORMALITY,NORMAL,6.497892,0.099092,0.311339,0.006224,0.003333
2,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CEN...,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CENTER,FAM_NORMALITY,NORMAL,6.717518,0.044001,0.135149,0.002553,0.000354
3,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CEN...,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CENTER,FAM_NORMALITY,NORMAL,6.841824,0.029337,0.089958,0.001698,0.000167
4,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CEN...,SCEN_CENTER_NORMAL_BASELINE_NORMALITY_GEST_CENTER,FAM_NORMALITY,NORMAL,6.928165,0.020879,0.063932,0.001203,0.000126


Non-sensor metadata in model feature list: []


## F4-F6 — build one outer fold, then make development-only OOF predictions

The next cells demonstrate the actual model sequence on the first configured outer fold. The validation predictions come only from the fold's training scenarios. They are used to compare calibration methods and select the 80% target-specific threshold; the held-out topology is not used for those choices.

In [8]:
train_manifest, train_features = build_window_dataset(
    train_records,
    window_size=int(config["window_size_samples"]),
    stride=int(config["stride_samples"]),
)
test_manifest, test_features = build_window_dataset(
    test_records,
    window_size=int(config["window_size_samples"]),
    stride=int(config["stride_samples"]),
)
display(train_features.groupby("label_mapped").size().rename("train_windows").to_frame())
display(test_features.groupby("label_mapped").size().rename("test_windows").to_frame())
print(f"Training topologies: {sorted(train_features['topology'].astype(str).unique())}")
print(f"Test topologies: {sorted(test_features['topology'].astype(str).unique())}")

,train_windows
label_mapped,
COMPRESSOR_DELAY,30
COMPRESSOR_FAULT,1328
LEAK,6460
NORMAL,666


,test_windows
label_mapped,
COMPRESSOR_DELAY,15
COMPRESSOR_FAULT,664
LEAK,3067
NORMAL,333


Training topologies: ['Center', 'Unknown', 'Upstream']
Test topologies: ['Downstream']


In [9]:
oof = inner_oof_predictions(
    train_features,
    random_state=int(config["random_state"]),
    k=int(config["d_neighbors"]),
)
oof_probability_columns = [f"p_raw_{label}" for label in TARGET_CLASSES]
display(oof[[
    "window_id", "Scenario_Family_ID", "topology", "label_mapped",
    *oof_probability_columns, "D", "support_n_families"
]].head())
print(f"OOF rows: {len(oof)}; any missing probability: {oof[oof_probability_columns].isna().any().any()}")

,window_id,Scenario_Family_ID,topology,label_mapped,p_raw_NORMAL,p_raw_LEAK,p_raw_COMPRESSOR_FAULT,p_raw_COMPRESSOR_DELAY,D,support_n_families
0,SCEN_CENTER_INITIAL_RAMP_UP_LEAKG3_w0002,FAM_LEAK_G3,Center,LEAK,0.067229,0.648334,0.197770,0.086667,1.0,1
1,SCEN_CENTER_INITIAL_RAMP_UP_LEAKG3_w0003,FAM_LEAK_G3,Center,LEAK,0.058960,0.749960,0.117746,0.073333,1.0,1
2,SCEN_CENTER_INITIAL_RAMP_UP_LEAKG3_w0004,FAM_LEAK_G3,Center,LEAK,0.049407,0.788819,0.088440,0.073333,1.0,1
3,SCEN_CENTER_INITIAL_RAMP_UP_LEAKG3_w0005,FAM_LEAK_G3,Center,LEAK,0.041316,0.804105,0.081246,0.073333,1.0,1
4,SCEN_CENTER_INITIAL_RAMP_UP_LEAKG3_w0006,FAM_LEAK_G3,Center,LEAK,0.042644,0.797154,0.086868,0.073333,1.0,1


OOF rows: 8484; any missing probability: False


## F6 — compare and select calibration

Classwise sigmoid and isotonic calibrators are compared with no calibration using inner-OOF log loss. The winning mapping is fitted on those development OOF rows and applied to the outer-test probabilities. The development comparison is not an independent calibration-validation estimate.

In [10]:
calibrator, selected_calibration, calibration_candidates = _oof_calibration_choice(oof)
display(pd.DataFrame(calibration_candidates))
print(f"Selected on inner OOF only: {selected_calibration}")

,calibration_method,development_log_loss,development_brier,development_ece,evaluation_data,selection_data
0,none,2.884122,0.901900,0.473612,inner_grouped_oof_fit_and_selection,inner_grouped_oof_only
1,sigmoid,0.499023,0.268494,0.052042,inner_grouped_oof_fit_and_selection,inner_grouped_oof_only
2,isotonic,0.402597,0.231016,0.114389,inner_grouped_oof_fit_and_selection,inner_grouped_oof_only


Selected on inner OOF only: isotonic


## F5-F6 — fit the outer-training baseline and predict the held-out fold

The estimator is the frozen 300-tree random forest. Window weights give each family equal total contribution before class balancing. The outer-test rows are only used to calculate evaluation results.

In [11]:
model = fit_baseline(
    train_features,
    random_state=int(config["random_state"]) + FOLD_ID,
)
raw_probabilities = aligned_probabilities(model, test_features)
calibrated_probabilities = calibrator.transform(raw_probabilities)
raw_predictions = np.asarray(TARGET_CLASSES)[raw_probabilities.argmax(axis=1)]
calibrated_predictions = np.asarray(TARGET_CLASSES)[calibrated_probabilities.argmax(axis=1)]
y_true = test_features["label_mapped"].astype(str).to_numpy()
print("Raw baseline:")
print(classification_report(y_true, raw_predictions, labels=list(TARGET_CLASSES), zero_division=0))
print("After selected calibration:")
print(classification_report(y_true, calibrated_predictions, labels=list(TARGET_CLASSES), zero_division=0))
display(pd.Series({
    "raw_accuracy": accuracy_score(y_true, raw_predictions),
    "raw_macro_f1": f1_score(y_true, raw_predictions, labels=list(TARGET_CLASSES), average="macro", zero_division=0),
    **probability_metrics(calibrated_probabilities, y_true),
}))

Raw baseline:
                  precision    recall  f1-score   support

          NORMAL       0.00      0.00      0.00       333
            LEAK       0.90      1.00      0.95      3067
COMPRESSOR_FAULT       1.00      1.00      1.00       664
COMPRESSOR_DELAY       1.00      1.00      1.00        15

        accuracy                           0.92      4079
       macro avg       0.72      0.75      0.74      4079
    weighted avg       0.84      0.92      0.88      4079

After selected calibration:
                  precision    recall  f1-score   support

          NORMAL       0.00      0.00      0.00       333
            LEAK       0.90      1.00      0.95      3067
COMPRESSOR_FAULT       1.00      1.00      1.00       664
COMPRESSOR_DELAY       0.94      1.00      0.97        15

        accuracy                           0.92      4079
       macro avg       0.71      0.75      0.73      4079
    weighted avg       0.84      0.92      0.88      4079



raw_accuracy                                                 0.918362
raw_macro_f1                                                 0.736824
multiclass_brier                                             0.217123
log_loss                                                     1.600538
ece                                                          0.135418
bin_counts                      [0, 0, 0, 4, 0, 542, 135, 1, 0, 3397]
bin_confidences     [0.05, 0.15000000000000002, 0.25, 0.3901046333...
bin_accuracies      [0.0, 0.0, 0.0, 0.75, 0.0, 0.9981549815498155,...
dtype: object

## F7 — compute family-excluded support D(x)

For each query family, every reference window from that family is removed before fitting the scaler as well as before finding up to 25 neighbors. This prevents the query family's training features from influencing its own support geometry. The output shows support proportions, the D margin, neighbor/family counts, and missing classes.

In [12]:
support = compute_empirical_support(
    test_features,
    train_features,
    k=int(config["d_neighbors"]),
)
display(pd.concat([
    test_features[["window_id", "Scenario_Family_ID", "label_mapped"]],
    support,
], axis=1).head())

,window_id,Scenario_Family_ID,label_mapped,q_NORMAL,q_LEAK,q_COMPRESSOR_FAULT,q_COMPRESSOR_DELAY,D,support_n_neighbors,support_n_families,support_missing_classes,query_family_excluded
0,SCEN_DOWNSTREAM_INITIAL_RAMP_UP_LEAKG3_w0002,FAM_LEAK_G3,LEAK,0.50,0.0,0.50,0.0,1.110223e-16,25,2,LEAK;COMPRESSOR_DELAY,True
1,SCEN_DOWNSTREAM_INITIAL_RAMP_UP_LEAKG3_w0003,FAM_LEAK_G3,LEAK,0.50,0.0,0.50,0.0,1.110223e-16,25,2,LEAK;COMPRESSOR_DELAY,True
2,SCEN_DOWNSTREAM_INITIAL_RAMP_UP_LEAKG3_w0004,FAM_LEAK_G3,LEAK,0.50,0.0,0.50,0.0,0.000000e+00,25,2,LEAK;COMPRESSOR_DELAY,True
3,SCEN_DOWNSTREAM_INITIAL_RAMP_UP_LEAKG3_w0005,FAM_LEAK_G3,LEAK,0.25,0.5,0.25,0.0,2.500000e-01,25,4,COMPRESSOR_DELAY,True
4,SCEN_DOWNSTREAM_INITIAL_RAMP_UP_LEAKG3_w0006,FAM_LEAK_G3,LEAK,0.20,0.6,0.20,0.0,4.000000e-01,25,5,COMPRESSOR_DELAY,True


## F8-F9 — calculate comparator scores and apply the three-outcome policy

B/B2/B3 are confidence, margin, and entropy-derived scores; C-D combines probability indicators without D; C adds D; D-only uses empirical support. The specific threshold is selected from calibrated inner-OOF C scores. The policy then returns `SPECIFIC`, `AMBIGUITY_SET`, or `UNRESOLVED`.

In [13]:
test_indicators = add_distinguishability_score(
    probability_indicators(calibrated_probabilities), support
)
oof_raw = oof[oof_probability_columns].to_numpy(dtype=float)
oof_calibrated = calibrator.transform(oof_raw)
oof_indicators = add_distinguishability_score(
    probability_indicators(oof_calibrated), oof
)
thresholds = {
    method: threshold_for_coverage(oof_indicators[column], TARGET_COVERAGE)
    for method, column in SCORE_COLUMNS.items()
}
policy = apply_diagnostic_policy(
    calibrated_probabilities,
    support,
    test_indicators["score_C"],
    specific_threshold=thresholds["C"],
    ambiguity_delta=float(config["policy"]["ambiguity_delta"]),
    max_candidates=int(config["policy"]["max_candidates"]),
    minimum_confidence_for_ambiguity=float(config["policy"]["minimum_confidence_for_ambiguity"]),
    minimum_support_for_ambiguity=float(config["policy"]["minimum_support_for_ambiguity"]),
)
display(policy.groupby("policy_outcome").size().rename("test windows").to_frame())
display(policy.head())

,test windows
policy_outcome,
AMBIGUITY_SET,1747
SPECIFIC,2331
UNRESOLVED,1


,policy_outcome,ambiguity_set,reason_code,candidate_count,candidates_truncated,specific_threshold,support_strength,support_missing_classes,support_n_neighbors,support_n_families
0,AMBIGUITY_SET,LEAK;COMPRESSOR_FAULT;NORMAL,MULTIPLE_SUPPORTED_CANDIDATES,3,False,0.652922,0.5,LEAK;COMPRESSOR_DELAY,25,2
1,AMBIGUITY_SET,LEAK;COMPRESSOR_FAULT;NORMAL,MULTIPLE_SUPPORTED_CANDIDATES,3,False,0.652922,0.5,LEAK;COMPRESSOR_DELAY,25,2
2,AMBIGUITY_SET,LEAK;NORMAL;COMPRESSOR_FAULT,MULTIPLE_SUPPORTED_CANDIDATES,3,False,0.652922,0.5,LEAK;COMPRESSOR_DELAY,25,2
3,SPECIFIC,,SPECIFIC_THRESHOLD_AND_SINGLE_CANDIDATE,1,False,0.652922,0.5,COMPRESSOR_DELAY,25,4
4,SPECIFIC,,SPECIFIC_THRESHOLD_AND_SINGLE_CANDIDATE,1,False,0.652922,0.6,COMPRESSOR_DELAY,25,5


## F10 — compare the six scores at matched coverage

This is a one-outer-fold diagnostic. The full report aggregates all three supplied outer folds. Matched-coverage selection ranks test rows by scores only; labels are used only to calculate risk after selection.

In [14]:
one_fold_comparison = []
for method, score_column in SCORE_COLUMNS.items():
    metrics = _selective_metrics(
        y_true,
        calibrated_predictions,
        test_indicators[score_column],
        test_features["window_id"],
        TARGET_COVERAGE,
    )
    one_fold_comparison.append({"method": method, **metrics})
display(pd.DataFrame(one_fold_comparison).sort_values("selective_risk"))

,method,observations,accepted,coverage,selective_risk,accepted_errors
5,D_only,4079,3264,0.800196,0.099265,324
0,B,4079,3264,0.800196,0.101103,330
2,C,4079,3264,0.800196,0.101103,330
1,C_minus_D,4079,3264,0.800196,0.101103,330
3,B2,4079,3264,0.800196,0.101103,330
4,B3,4079,3264,0.800196,0.101103,330


## Repeat model fitting and testing for all three folds

The earlier cells trace Fold 1 in detail. This cell independently loads, featurizes, calibrates, and fits the frozen model for every outer fold, then evaluates each held-out topology. Calibration is selected from that fold's training-only OOF predictions; test labels are used only for the final reports.

In [15]:
from sklearn.metrics import balanced_accuracy_score

all_fold_model_summary = []
all_fold_raw_confusions = {}
all_fold_calibrated_confusions = {}
for fold_id in map(int, config["outer_folds"]):
    fold_train_records, fold_test_records, fold_diagnostics = load_outer_fold(fold_id)
    _, fold_train_features = build_window_dataset(
        fold_train_records,
        window_size=int(config["window_size_samples"]),
        stride=int(config["stride_samples"]),
    )
    _, fold_test_features = build_window_dataset(
        fold_test_records,
        window_size=int(config["window_size_samples"]),
        stride=int(config["stride_samples"]),
    )
    fold_oof = inner_oof_predictions(
        fold_train_features,
        random_state=int(config["random_state"]),
        k=int(config["d_neighbors"]),
    )
    fold_calibrator, fold_calibration_method, _ = _oof_calibration_choice(fold_oof)
    fold_model = fit_baseline(
        fold_train_features,
        random_state=int(config["random_state"]) + fold_id,
    )
    fold_y_true = fold_test_features["label_mapped"].astype(str).to_numpy()
    fold_raw_probabilities = aligned_probabilities(fold_model, fold_test_features)
    fold_calibrated_probabilities = fold_calibrator.transform(fold_raw_probabilities)
    fold_raw_predictions = np.asarray(TARGET_CLASSES)[fold_raw_probabilities.argmax(axis=1)]
    fold_calibrated_predictions = np.asarray(TARGET_CLASSES)[fold_calibrated_probabilities.argmax(axis=1)]
    all_fold_model_summary.append({
        "fold": fold_id,
        "test_topology": ";".join(fold_diagnostics["test_topologies"]),
        "train_scenarios": len(fold_train_records),
        "test_scenarios": len(fold_test_records),
        "train_windows": len(fold_train_features),
        "test_windows": len(fold_test_features),
        "raw_accuracy": accuracy_score(fold_y_true, fold_raw_predictions),
        "raw_macro_f1": f1_score(fold_y_true, fold_raw_predictions, labels=list(TARGET_CLASSES), average="macro", zero_division=0),
        "raw_balanced_accuracy": balanced_accuracy_score(fold_y_true, fold_raw_predictions),
        "calibrated_accuracy": accuracy_score(fold_y_true, fold_calibrated_predictions),
        "calibration_selected_on_training_oof": fold_calibration_method,
    })
    all_fold_raw_confusions[fold_id] = pd.crosstab(
        pd.Categorical(fold_y_true, categories=TARGET_CLASSES),
        pd.Categorical(fold_raw_predictions, categories=TARGET_CLASSES),
        rownames=["actual"], colnames=["predicted"], dropna=False,
    )
    all_fold_calibrated_confusions[fold_id] = pd.crosstab(
        pd.Categorical(fold_y_true, categories=TARGET_CLASSES),
        pd.Categorical(fold_calibrated_predictions, categories=TARGET_CLASSES),
        rownames=["actual"], colnames=["predicted"], dropna=False,
    )

display(pd.DataFrame(all_fold_model_summary))
for fold_id in map(int, config["outer_folds"]):
    print(f"\nFold {fold_id} raw predictions")
    display(all_fold_raw_confusions[fold_id])
    print(f"Fold {fold_id} after calibration")
    display(all_fold_calibrated_confusions[fold_id])

,fold,test_topology,train_scenarios,test_scenarios,train_windows,test_windows,raw_accuracy,raw_macro_f1,raw_balanced_accuracy,calibrated_accuracy,calibration_selected_on_training_oof
0,1,Downstream,31,15,8484,4079,0.918362,0.736824,0.750000,0.918362,isotonic
1,2,Center,31,15,8484,4079,0.196127,0.234092,0.215419,0.162785,isotonic
2,3,Upstream,31,15,8484,4079,1.000000,1.000000,1.000000,0.166462,sigmoid



Fold 1 raw predictions


predicted,NORMAL,LEAK,COMPRESSOR_FAULT,COMPRESSOR_DELAY
actual,,,,
NORMAL,0,331,2,0
LEAK,0,3067,0,0
COMPRESSOR_FAULT,0,0,664,0
COMPRESSOR_DELAY,0,0,0,15


Fold 1 after calibration


predicted,NORMAL,LEAK,COMPRESSOR_FAULT,COMPRESSOR_DELAY
actual,,,,
NORMAL,0,331,1,1
LEAK,0,3067,0,0
COMPRESSOR_FAULT,0,0,664,0
COMPRESSOR_DELAY,0,0,0,15



Fold 2 raw predictions


... (3 outputs left out)


Fold 3 raw predictions


predicted,NORMAL,LEAK,COMPRESSOR_FAULT,COMPRESSOR_DELAY
actual,,,,
NORMAL,333,0,0,0
LEAK,0,3067,0,0
COMPRESSOR_FAULT,0,0,664,0
COMPRESSOR_DELAY,0,0,0,15


Fold 3 after calibration


predicted,NORMAL,LEAK,COMPRESSOR_FAULT,COMPRESSOR_DELAY
actual,,,,
NORMAL,0,332,0,1
LEAK,3066,0,1,0
COMPRESSOR_FAULT,0,0,664,0
COMPRESSOR_DELAY,0,0,0,15


## F11 — inspect all-fold family-level uncertainty

The official test resamples and sign-flips family-level risks rather than individual windows. The number of independent family units is small, so the results are exploratory; confidence intervals spanning zero do not establish superiority.

In [16]:
for filename in ("BASELINE_RESULTS.csv", "SELECTIVE_EVALUATION_RESULTS.csv",
                 "STATISTICAL_VALIDATION.csv", "PER_FAMILY_RESULTS.csv"):
    path = RESULTS_ROOT / filename
    print(f"\n{filename}")
    if path.exists():
        table = pd.read_csv(path)
        display(table.head(12))
    else:
        print("Not generated yet; run the full experiment cell below.")


BASELINE_RESULTS.csv


,outer_fold,scenarios_train,scenarios_test,windows_train,windows_test,raw_accuracy,raw_macro_f1,raw_balanced_accuracy,calibrated_accuracy,calibration_method_selected_on,selected_calibration_method,train_test_family_overlap_count,evaluation_estimand
0,1,31,15,8484,4079,0.918362,0.736824,0.750000,0.918362,inner_grouped_oof_only,isotonic,9,cross_topology_transfer
1,2,31,15,8484,4079,0.196127,0.234092,0.215419,0.162785,inner_grouped_oof_only,isotonic,9,cross_topology_transfer
2,3,31,15,8484,4079,1.000000,1.000000,1.000000,0.166462,inner_grouped_oof_only,sigmoid,9,cross_topology_transfer



SELECTIVE_EVALUATION_RESULTS.csv


,outer_fold,method,target_coverage,matched_coverage,matched_selective_risk,matched_accepted,development_threshold,development_threshold_test_coverage,development_threshold_test_selective_risk,family_overlap_count,evaluation_estimand
0,1,B,0.8,0.800196,0.101103,3264,0.740388,0.832802,0.097145,9,cross_topology_transfer
1,1,C_minus_D,0.8,0.800196,0.101103,3264,0.593335,0.832802,0.097145,9,cross_topology_transfer
2,1,C,0.8,0.800196,0.101103,3264,0.652922,0.832312,0.097202,9,cross_topology_transfer
3,1,B2,0.8,0.800196,0.101103,3264,0.568214,0.832802,0.097145,9,cross_topology_transfer
4,1,B3,0.8,0.800196,0.101103,3264,0.530687,0.865408,0.093768,9,cross_topology_transfer
5,1,D_only,0.8,0.800196,0.099265,3264,1.000000,0.946310,0.083938,9,cross_topology_transfer
6,2,B,0.8,0.800196,0.941789,3264,0.879934,0.907330,0.922724,9,cross_topology_transfer
7,2,C_minus_D,0.8,0.800196,0.941789,3264,0.791675,0.907330,0.922724,9,cross_topology_transfer
8,2,C,0.8,0.800196,0.944547,3264,0.744058,0.912969,0.917025,9,cross_topology_transfer
9,2,B2,0.8,0.800196,0.941789,3264,0.759867,0.907330,0.922724,9,cross_topology_transfer



STATISTICAL_VALIDATION.csv


,comparison,independent_units,mean_family_risk_difference,bootstrap_ci_95_low,bootstrap_ci_95_high,two_sided_sign_flip_p,permutation_method,bootstrap_replicates,interpretation
0,C_minus_C_minus_D,9,-0.011795,-0.039132,0.003376,0.8125,exact_family_sign_flip,1000,negative favors method C
1,C_minus_B,9,-0.011926,-0.039403,0.003340,0.7500,exact_family_sign_flip,1000,negative favors method C



PER_FAMILY_RESULTS.csv


,outer_fold,Scenario_Family_ID,method,target_coverage,observations,accepted,coverage,selective_risk,accepted_errors
0,1,FAM_COMPRESSOR_DELAY_C1,B,0.8,15,12,0.800000,0.0,0
1,1,FAM_COMPRESSOR_DELAY_C1,C_minus_D,0.8,15,12,0.800000,0.0,0
2,1,FAM_COMPRESSOR_DELAY_C1,C,0.8,15,12,0.800000,0.0,0
3,1,FAM_COMPRESSOR_DELAY_C1,B2,0.8,15,12,0.800000,0.0,0
4,1,FAM_COMPRESSOR_DELAY_C1,B3,0.8,15,12,0.800000,0.0,0
5,1,FAM_COMPRESSOR_DELAY_C1,D_only,0.8,15,12,0.800000,0.0,0
6,1,FAM_COMPRESSOR_FAULT_C1,B,0.8,664,532,0.801205,0.0,0
7,1,FAM_COMPRESSOR_FAULT_C1,C_minus_D,0.8,664,532,0.801205,0.0,0
8,1,FAM_COMPRESSOR_FAULT_C1,C,0.8,664,532,0.801205,0.0,0
9,1,FAM_COMPRESSOR_FAULT_C1,B2,0.8,664,532,0.801205,0.0,0


## F12-F13 — verify outputs and reproducibility

The command-line runner writes the canonical all-fold results, Stage 2 figures, hashes, Git revision, Python version, and installed dependency versions. The existing dashboard/demo figures are not overwritten. The focused test outcome is recorded separately.

In [17]:
provenance_path = RESULTS_ROOT / "RESULTS_PROVENANCE.csv"
if provenance_path.exists():
    provenance = pd.read_csv(provenance_path)
    display(provenance.groupby("artifact_type").size().rename("entries").to_frame())
    display(provenance[provenance["artifact_type"] == "runtime_dependency"][["path", "version"]])
    print(f"Python: {provenance['python_version'].dropna().iloc[0]}")
else:
    print("No provenance yet; run the full experiment cell.")

test_results_path = PROJECT_ROOT / "04_validation" / "TEST_RESULTS.md"
if test_results_path.exists():
    print(test_results_path.read_text(encoding="utf-8"))

,entries
artifact_type,
config,1
input_parquet,46
manifest,1
outer_folds,1
runtime_dependency,12
source,4


,path,version
53,numpy,2.2.1
54,pandas,2.3.0
55,scipy,1.16.0
56,pyarrow,26.0.1
57,fastparquet,2026.9.0
58,PyYAML,6.0.3
59,matplotlib,3.10.3
60,seaborn,0.13.2
61,PyWavelets,1.9.0
62,scikit-learn,1.7.1


Python: 3.11.9
# Faisal Pipeline Test Results

## Focused automated tests

Command:

```text
python -m pytest tests/test_faisal_pipeline.py tests/test_data_loader.py tests/test_dataset_audit.py -q
```

Result: **20 passed**. This covers the new config-version guard,
preprocessing, event-boundary,
incomplete-tail, family-excluded support, row-order permutation, calibration,
policy, the query-family scaler-exclusion regression, data-loader, and
dataset-audit tests.

Import check: `h2prism.experiments` and `h2prism.evaluation` imported
successfully with the repository's `src` directory on `sys.path`, as configured
by the CLI and test setup.

Notebook validation: the `.ipynb` JSON and all code-cell syntax were checked;
all 18 code cells executed successfully for the one-fold walkthrough
(8,484 train windows, 4,079 test windows, isotonic calibration selected). Fold
diagnostics are derived from the supplied fold table, so the walkthrough does
not require a previously generated report with a 

## Optional — rerun the accepted three-fold experiment

The preceding cells fit and inspect the first outer fold. Set `RUN_FULL_OUTER_EXPERIMENT` to `True` to regenerate the complete accepted run with the frozen config. This takes several minutes and rewrites only the Faisal-generated CSVs, report, provenance, and `_stage2.png` figures.

In [18]:
RUN_FULL_OUTER_EXPERIMENT = False
if RUN_FULL_OUTER_EXPERIMENT:
    generated_artifacts = run_experiments(DEFAULT_CONFIG, DEFAULT_OUTPUT_DIR)
    display(pd.Series({name: str(path) for name, path in generated_artifacts.items()}))
else:
    print("Set RUN_FULL_OUTER_EXPERIMENT = True and rerun this cell to regenerate all folds.")

Set RUN_FULL_OUTER_EXPERIMENT = True and rerun this cell to regenerate all folds.


After a full rerun, this final cell reloads the generated all-fold tables so the notebook view reflects the new run rather than cached results from earlier cells.

In [19]:
if RUN_FULL_OUTER_EXPERIMENT:
    refreshed = pd.read_csv(RESULTS_ROOT / "SELECTIVE_EVALUATION_RESULTS.csv")
    display(refreshed.groupby("method").agg(
        folds=("outer_fold", "nunique"),
        matched_coverage=("matched_coverage", "mean"),
        matched_risk=("matched_selective_risk", "mean"),
    ))
    display(pd.read_csv(RESULTS_ROOT / "STATISTICAL_VALIDATION.csv"))
    display(pd.read_csv(RESULTS_ROOT / "FOLD_VALIDATION_REPORT.csv"))
else:
    print("The generated tables above are the current accepted run.")

The generated tables above are the current accepted run.
